# Explore SMLM simulations

Use this notebook to explore spot density, brightness, background and noise, compare them with your recordings, and try DECODE’s own simulator. Run cells from top to bottom. Edit the values in the parameter cells and rerun the following cells. No training happens here.

**Two simulators:** the first is a simple educational Gaussian model that runs in the project environment; it is not DECODE and is not a calibrated training dataset. The optional final section uses DECODE and a spline calibration. Numerical values below are illustrative unless explicitly identified as measured.


## What is a PSF and how do I get one?

The point spread function (PSF) describes the spot produced by a single point-like emitter. Its shape depends on the optics, colour/channel, depth and alignment. DECODE uses a fitted **3D spline PSF** to render realistic spots at different depths. A good calibration is specific to the imaging setup and channel; 117 nm/pixel alone cannot specify its shape.

Ask the person who collected the data for: (1) a fluorescent bead z-stack acquired on the same microscope, objective, optical path/channel and comparable imaging conditions, with its z step and units; or (2) an existing SMAP/DECODE `3dcal.mat` spline calibration for those data. Ask which channel and orientation it belongs to and whether it has already been transformed. Also request pixel size, camera offset/baseline, electrons per ADU, read noise, exposure time, fluorophore and acquisition settings. Check 117 nm/pixel against each recording’s metadata.

The `experimental_data_workflow/beads` and `spline_calibration_3dcal.mat` files are **DECODE’s example**, not a calibration of this Nanoimager. They let you learn the workflow, not establish accuracy on your own images. If no relevant calibration exists, ask whether the microscope can be calibrated with a new bead z-stack; a retrospective calibration might differ from the original experiment. DECODE’s [tutorial](https://decode.readthedocs.io/en/release-0.10/tutorial.html) describes fitting the bead stack with SMAP, including for 2D work. A Gaussian PSF can be useful for exploration, but is not a substitute for validating a calibrated model.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Find the project root whether the kernel starts in the project or a subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preprocessing').is_dir()), None)
if root is None:
    raise FileNotFoundError('Open this notebook from inside the mphys project folder.')
print('Project root:', root)


## 1. A simplified, editable simulator

Each frame has randomly placed Gaussian spots, a uniform background, Poisson shot noise and independent Gaussian read noise. Brightness and background are in **detected electrons**; the output is in ADU (camera counts). The `read_noise_e` default of 1.6 is the Nanoimager brochure’s nominal RMS read noise, not an individual camera calibration. The other camera and emitter values are made-up examples. A constant read noise does not capture pixel-dependent sCMOS properties; spots are 2D Gaussian, not a measured 3D PSF.

Change one value at a time. `emitters_per_frame` is for the whole simulated image, so compare densities only for equal image sizes. `psf_sigma_px` is the Gaussian standard deviation, not resolution; FWHM ≈ 2.355 × sigma. A new random seed makes a different frame.


In [ ]:
# EXPLORATORY values, not calibrated Nanoimager settings.
image_size = 128                # pixels per side
pixel_size_nm = 117.0           # confirmed in two low-density acquisition files
emitters_per_frame = 30         # average, sampled from a Poisson distribution
mean_signal_e = 2500.0          # total detected electrons per emitter on average
background_e = 30.0             # mean detected background electrons per pixel
psf_sigma_px = 1.2              # Gaussian width, illustrative only
read_noise_e = 1.6              # brochure nominal RMS; verify for your camera
electrons_per_adu = 1.0         # PLACEHOLDER: obtain camera conversion factor
baseline_adu = 100.0            # PLACEHOLDER: obtain camera offset
seed = 42


In [ ]:
if image_size < 8 or emitters_per_frame < 0 or min(mean_signal_e, background_e, read_noise_e) < 0 or psf_sigma_px <= 0 or electrons_per_adu <= 0:
    raise ValueError('Check image size and non-negative signal, background, noise and positive sigma/gain.')
rng = np.random.default_rng(seed)
expected_e = np.full((image_size, image_size), background_e, dtype=float)
count = rng.poisson(emitters_per_frame)
positions = rng.uniform(0, image_size, (count, 2))  # y, x in pixels
photons = rng.exponential(mean_signal_e, count)
radius = int(np.ceil(4 * psf_sigma_px))
for (y, x), brightness in zip(positions, photons):
    y0, y1 = max(0, int(y)-radius), min(image_size, int(y)+radius+1)
    x0, x1 = max(0, int(x)-radius), min(image_size, int(x)+radius+1)
    yy, xx = np.mgrid[y0:y1, x0:x1]
    # Continuous Gaussian normalization: edge spots lose light outside the frame.
    expected_e[y0:y1, x0:x1] += brightness * np.exp(-((xx-x)**2+(yy-y)**2)/(2*psf_sigma_px**2)) / (2*np.pi*psf_sigma_px**2)
shot_e = rng.poisson(expected_e)
noisy_e = shot_e + rng.normal(0, read_noise_e, expected_e.shape)
simulated_adu = np.maximum(0, baseline_adu + noisy_e / electrons_per_adu)
print(f'{count} emitters; field: {image_size*pixel_size_nm/1000:.2f} µm square')
print('Output ADU percentiles (1, 50, 99, 99.9):', np.percentile(simulated_adu, [1, 50, 99, 99.9]).round(1))
fig, ax = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
a = ax[0].imshow(expected_e, cmap='gray'); ax[0].set_title('Expected signal before noise (electrons)'); plt.colorbar(a, ax=ax[0])
b = ax[1].imshow(simulated_adu, cmap='gray'); ax[1].set_title('Simulated camera frame (ADU)'); plt.colorbar(b, ax=ax[1])
plt.show()


## 2. Compare with your raw data

Choose an existing **raw** TIFF and channel. This reads one frame only, then compares a central crop with the simulation using the **same ADU display limits**. The simulated ADUs are arbitrary until camera conversion and offset are measured, so this is currently a visual exercise rather than a quantitative match. Use the same limits for low- and high-density files. Leave `real_tif` as `None` to skip. Registered/drift-corrected frames have interpolated noise and should not be assumed equivalent to raw frames.


In [ ]:
real_tif = None  # e.g. root / 'high_density_examples' / 'your_file.tif'
frame_index = 0
channel = 'left'  # 'left', 'right', or 'full' for a single-channel TIFF
display_min, display_max = 100.0, 500.0  # fixed ADU limits; adjust for both datasets

if real_tif is None:
    print('Set real_tif to a recording path to enable comparison.')
else:
    if channel not in ('left', 'right', 'full') or display_max <= display_min:
        raise ValueError('Check channel and display limits.')
    with Image.open(real_tif) as movie:
        movie.seek(frame_index)
        real = np.asarray(movie).copy()
    if real.ndim != 2:
        raise ValueError('Expected a grayscale TIFF frame.')
    if channel != 'full':
        if real.shape[1] % 2: raise ValueError('Expected two equal-width channels.')
        mid = real.shape[1] // 2
        real = real[:, :mid] if channel == 'left' else real[:, mid:]
    h, w = real.shape
    n = min(image_size, h, w)
    crop = real[(h-n)//2:(h-n)//2+n, (w-n)//2:(w-n)//2+n]
    print('Real crop ADU percentiles (1, 50, 99, 99.9):', np.percentile(crop, [1, 50, 99, 99.9]).round(1))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, data, title in zip(axes, (crop, simulated_adu), ('Real raw crop', 'Simplified simulation')):
        im = ax.imshow(data, cmap='gray', vmin=display_min, vmax=display_max); ax.set_title(title)
    fig.colorbar(im, ax=axes, label='ADU (same scale)'); plt.show()


## 3. Optional: DECODE’s actual spline simulator

This section needs a working DECODE installation **including its compiled spline dependency**; the ordinary `.venv` does not have it by default. Select that environment as the notebook kernel. DECODE’s published macOS installation is CPU-only and its older dependencies may not resolve on Apple Silicon. See [installation](https://decode.readthedocs.io/en/release-0.10/installation.html).

The following starts from the bundled example’s parameters and calibration, producing **example-data simulations**, not a Nanoimager-calibrated simulation. It uses the example’s EMCCD camera settings, even though your microscope uses sCMOS. To simulate your data, replace the calibration, camera parameters, emitter/background settings and scaling with validated values. DECODE’s training notebook says to set `param.Camera.em_gain = None` for sCMOS; its simple training camera model uses uniform Gaussian read noise, not a full per-pixel noise map. Carefully check units and compare real and simulated frames. The example’s original calibration path is overridden locally; no files are overwritten.


In [ ]:
run_decode = False  # Change to True only in a working DECODE environment.

if run_decode:
    import torch
    import decode
    import decode.utils.param_io
    import decode.neuralfitter.train.random_simulation
    param = decode.utils.param_io.load_params(str(root / 'experimental_data_workflow' / 'param_run.yaml'))
    param.InOut.calibration_file = str(root / 'experimental_data_workflow' / 'spline_calibration_3dcal.mat')
    param.Hardware.device = 'cpu'
    param.Hardware.device_simulation = 'cpu'
    # Edit these example values to explore; do not treat them as Nanoimager calibration.
    param.Simulation.emitter_av = 25
    param.Simulation.bg_uniform = [20.0, 200.0]  # expected photon background range
    param.Simulation.intensity_mu_sig = [7000.0, 3000.0]
    # Limit sampling; DECODE's frame bounds are inclusive.
    param.HyperParameter.pseudo_ds_size = 7
    param.TestSet.test_size = 7
    torch.manual_seed(42)
    simulator, _ = decode.neuralfitter.train.random_simulation.setup_random_simulation(param)
    emitters, decode_frames, decode_background = simulator.sample()
    decode_frames = decode_frames.detach().cpu().numpy()
    print('Simulated frame shape:', decode_frames.shape, '; convert2photons:', param.Camera.convert2photons)
    plt.figure(figsize=(6, 6)); plt.imshow(decode_frames[0], cmap='gray'); plt.colorbar(); plt.title('DECODE example simulation'); plt.show()
else:
    print('Skipped DECODE section. Set run_decode = True after installing DECODE and spline.')


## What to do next

1. Compare raw low- and high-density frames on the same ADU scale. Note the frame, channel, exposure and display limits.
2. Obtain a suitable channel-specific calibration and camera settings; verify the instrument and acquisition metadata.
3. Compare calibrated DECODE simulations and real frames, including spot shapes, brightness distribution, background and noise. DECODE’s official training notebook includes a comparison in photon units.
4. Only then configure training and evaluate against suitable real data or an established method. This notebook does not train a model or establish localisation accuracy.
